# Colab: Experiments 1a, 1b, 2, 3, and 4 (model-selectable)

This notebook runs the current experiment pipelines on Colab with `meta-llama/Llama-3.2-3B-Instruct`, `meta-llama/Llama-3.2-3B`, or GPT-2 large.

Important notes:
- Supported choices are `Llama-3.2-3B-Instruct`, `Llama-3.2-3B`, and `gpt-2-large`.
- If you choose Llama, you must accept the Meta Llama license on Hugging Face and provide a Colab secret named `HF_TOKEN`.
- If you choose GPT-2, Hugging Face login is skipped.
- Optional core lexical-overlap comparison is supported for `1b` and Experiment 2 generation audit.
- Experiment 2 in this notebook uses the definitive full-sentence generation-audit path (former 2c).
- Experiment 3 and Experiment 4 are optional; enable them via `RUN_EXPERIMENT_3` / `RUN_EXPERIMENT_4`.
- Jabberwocky is still included inside those runs, but there is no separate Jabberwocky lexical-overlap manipulation here.


In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/Geometry-of-Syntax')
REPO_URL = 'https://github.com/<YOUR-USERNAME>/Geometry-of-Syntax.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/Geometry-of-Syntax
!git pull --ff-only || true
!nvidia-smi || true
!python --version

# Keep Colab's CUDA torch. Install only Python-side dependencies used by the repo.
!pip install -q pandas scipy statsmodels tqdm accelerate sentencepiece huggingface_hub pyyaml "transformers>=4.57,<5"


In [ ]:
from huggingface_hub import login

# Choose one: 'Llama-3.2-3B-Instruct', 'Llama-3.2-3B', or 'gpt-2-large'
MODEL_CHOICE = 'Llama-3.2-3B'

MODEL_OPTIONS = {
    'Llama-3.2-3B-Instruct': 'meta-llama/Llama-3.2-3B-Instruct',
    'Llama-3.2-3B': 'meta-llama/Llama-3.2-3B',
    # Keep user-facing name as requested; map to HF repo id internally.
    'gpt-2-large': 'gpt2-large',
}
if MODEL_CHOICE not in MODEL_OPTIONS:
    raise ValueError(f'Unknown MODEL_CHOICE: {MODEL_CHOICE}. Choose one of: {sorted(MODEL_OPTIONS)}')

MODEL_NAME = MODEL_OPTIONS[MODEL_CHOICE]

if MODEL_NAME.startswith('meta-llama/Llama-3.2-3B'):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None

    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login loaded from Colab secret HF_TOKEN.')
    else:
        print('No HF_TOKEN secret found. Add one in Colab after accepting the Meta Llama license, then rerun this cell.')
else:
    print(f'Selected {MODEL_CHOICE}; skipping Hugging Face login.')

print({'model_choice': MODEL_CHOICE, 'model_name': MODEL_NAME})


In [ ]:
DEVICE = 'cuda'
TORCH_DTYPE = 'float16'
SEED = 13
MAX_ITEMS = 2080
MAX_ITEMS_1A = 15000

RUN_EXPERIMENT_1A = False
RUN_EXPERIMENT_1B = True
RUN_EXPERIMENT_2 = True
RUN_EXPERIMENT_3 = False
RUN_EXPERIMENT_4 = False

# Base configs used to derive single-model runtime configs for Colab runs.
EXPERIMENT3_BASE_CONFIG = 'configs/experiment3_colab.yaml'
EXPERIMENT4_BASE_CONFIG = 'configs/exp4.yaml'

# Turn this on only if you explicitly want the contaminated core comparison too.
RUN_CORE_LEXICAL_OVERLAP_COMPARISON = False

# Conservative defaults for Colab GPU memory.
BATCH_SIZE_1A = 32
BATCH_SIZE_1B = 4
BATCH_SIZE_2 = 4
BATCH_SIZE_3 = 2
BATCH_SIZE_4 = 2
EXP2_MAX_NEW_TOKENS = 24


MODEL_TAG = MODEL_CHOICE.replace('.', '').replace('-', '_')
BASE_OUTPUT_ROOT = f'behavioral_results/colab_{MODEL_TAG}'

# Keep Exp3/Exp4 outputs inside model-specific Colab output root so zip includes them.
EXPERIMENT3_OUTPUT_DIR = f'{BASE_OUTPUT_ROOT}/experiment-3/exp3_run'
EXPERIMENT4_OUTPUT_DIR = f'{BASE_OUTPUT_ROOT}/experiment-4/exp4_run'
CORE_PRIME_MODES = ['lexically_controlled']
if RUN_CORE_LEXICAL_OVERLAP_COMPARISON:
    CORE_PRIME_MODES.append('lexical_overlap')

EXP1A_OUTPUT_ROOT = f'{BASE_OUTPUT_ROOT}/experiment-1/experiment-1a/transitive_token_profiles'

EXP1B_OUTPUT_ROOTS = {
    mode: f'{BASE_OUTPUT_ROOT}/experiment-1/experiment-1b/processing_experiment_1b_{MODEL_TAG}_v1_{mode}' for mode in CORE_PRIME_MODES
}
EXP1B_PRIME_CONDITIONS = ['active', 'passive', 'no_prime', 'filler']

# Definitive Experiment 2 (former 2c): full-sentence generation audit outputs.
EXP2_AUDIT_OUTPUT_ROOTS = {
    mode: f'{BASE_OUTPUT_ROOT}/experiment-2/experiment-2_generation_audit_{mode}' for mode in CORE_PRIME_MODES
}

# Best current Experiment 2 wording from the local prompt screen.
EXP2_EVENT_STYLE = 'involving_event'
EXP2_ROLE_STYLE = 'did_to'
EXP2_QUOTE_STYLE = 'mary_answered'

print({
    'model_choice': MODEL_CHOICE,
    'model_name': MODEL_NAME,
    'device': DEVICE,
    'torch_dtype': TORCH_DTYPE,
    'max_items': MAX_ITEMS,
    'max_items_1a': MAX_ITEMS_1A,
    'core_prime_modes': CORE_PRIME_MODES,
    'run_1a': RUN_EXPERIMENT_1A,
    'run_1b': RUN_EXPERIMENT_1B,
    'run_2': RUN_EXPERIMENT_2,
    'run_3': RUN_EXPERIMENT_3,
    'run_4': RUN_EXPERIMENT_4,
    'exp1a_output_root': EXP1A_OUTPUT_ROOT,
    'exp1b_output_roots': EXP1B_OUTPUT_ROOTS,
    'exp2_audit_output_roots': EXP2_AUDIT_OUTPUT_ROOTS,
    'exp3_base_config': EXPERIMENT3_BASE_CONFIG,
    'exp3_output_dir': EXPERIMENT3_OUTPUT_DIR,
    'exp4_base_config': EXPERIMENT4_BASE_CONFIG,
    'exp4_output_dir': EXPERIMENT4_OUTPUT_DIR,
})




In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)

def run_command(command):
    print('\nRunning:')
    print(' '.join(command))
    subprocess.run(command, check=True)

def zip_output_dir(path):
    path = Path(path).resolve()
    if not path.exists():
        print(f'Skipping zip; missing output dir: {path}')
        return None
    archive = Path(shutil.make_archive(str(path), 'zip', root_dir=str(path)))
    print(f'Zipped: {archive}')
    return archive


if RUN_EXPERIMENT_1A:
    run_command([
        sys.executable,
        'scripts/10_run_colab_experiment_1a.py',
        '--model-name', MODEL_NAME,
        '--device', DEVICE,
        '--batch-size', str(BATCH_SIZE_1A),
        '--max-items', str(MAX_ITEMS_1A),
        '--seed', str(SEED),
        '--output-root', EXP1A_OUTPUT_ROOT,
    ])
    zip_output_dir(Path(REPO_DIR) / EXP1A_OUTPUT_ROOT)
else:
    print('Skipping Experiment 1a.')

if RUN_EXPERIMENT_1B:
    exp1b_roots = []
    for core_mode in CORE_PRIME_MODES:
        which_for_mode = 'both' if core_mode == 'lexically_controlled' else 'core'
        run_command([
            sys.executable,
            'scripts/16_run_processing_experiment_1b.py',
            '--model-name', MODEL_NAME,
            '--device', DEVICE,
            '--torch-dtype', TORCH_DTYPE,
            '--batch-size', str(BATCH_SIZE_1B),
            '--max-items', str(MAX_ITEMS),
            '--which', which_for_mode,
            '--core-prime-mode', core_mode,
            '--seed', str(SEED),
            '--output-root', EXP1B_OUTPUT_ROOTS[core_mode],
            '--prime-conditions', *EXP1B_PRIME_CONDITIONS,
        ])
        exp1b_roots.append(Path(REPO_DIR) / EXP1B_OUTPUT_ROOTS[core_mode])

    for root in exp1b_roots:
        zip_output_dir(root)
else:
    print('Skipping Experiment 1b.')



In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
import yaml
import pandas as pd

os.chdir(REPO_DIR)

def run_command(command):
    print('\nRunning:')
    print(' '.join(command))
    subprocess.run(command, check=True)

def zip_output_dir(path):
    path = Path(path).resolve()
    if not path.exists():
        print(f'Skipping zip; missing output dir: {path}')
        return None
    archive = Path(shutil.make_archive(str(path), 'zip', root_dir=str(path)))
    print(f'Zipped: {archive}')
    return archive

def summarize_binary(frame, label_column):
    counts = (
        frame.groupby(['prompt_column', 'prime_condition', label_column], as_index=False)
        .agg(n_items=('item_index', 'count'))
    )
    totals = (
        frame.groupby(['prompt_column', 'prime_condition'], as_index=False)
        .agg(total_items=('item_index', 'count'))
    )
    merged = counts.merge(totals, on=['prompt_column', 'prime_condition'], how='left')
    merged['share'] = merged['n_items'] / merged['total_items']
    return merged.sort_values(['prompt_column', 'prime_condition', label_column]).reset_index(drop=True)

def summarize_quality(frame, label_column):
    annotated = frame.copy()
    annotated['is_active'] = annotated[label_column].astype(str).eq('active')
    annotated['is_passive'] = annotated[label_column].astype(str).eq('passive')
    annotated['is_other'] = annotated[label_column].astype(str).eq('other')
    annotated['is_congruent'] = (
        ((annotated['prime_condition'] == 'active') & annotated['is_active'])
        | ((annotated['prime_condition'] == 'passive') & annotated['is_passive'])
    )
    return (
        annotated.groupby(['prompt_column', 'prime_condition'], as_index=False)
        .agg(
            n_items=('item_index', 'count'),
            active_rate=('is_active', 'mean'),
            passive_rate=('is_passive', 'mean'),
            other_rate=('is_other', 'mean'),
            congruent_rate=('is_congruent', 'mean'),
        )
        .sort_values(['prompt_column', 'prime_condition'])
        .reset_index(drop=True)
    )

if RUN_EXPERIMENT_2:
    prompt_output_dir = Path(REPO_DIR) / 'corpora' / 'transitive'
    exp2_roots = []

    for core_mode in CORE_PRIME_MODES:
        exp2_root = Path(REPO_DIR) / EXP2_AUDIT_OUTPUT_ROOTS[core_mode]
        exp2_root.mkdir(parents=True, exist_ok=True)
        exp2_roots.append(exp2_root)

        # 1) Export deterministic prompt CSVs for the selected core mode.
        run_command([
            sys.executable,
            'scripts/28_export_demo_prompt_csvs.py',
            '--core-prime-mode', core_mode,
            '--max-items', str(MAX_ITEMS),
            '--seed', str(SEED),
            '--event-style', EXP2_EVENT_STYLE,
            '--role-style', EXP2_ROLE_STYLE,
            '--quote-style', EXP2_QUOTE_STYLE,
            '--output-dir', str(prompt_output_dir),
        ])

        core_prompt_csv = prompt_output_dir / f'experiment_2_core_demo_prompts_{core_mode}.csv'
        jabber_prompt_csv = prompt_output_dir / f'experiment_2_jabberwocky_demo_prompts_{core_mode}.csv'

        # 2) Generate full-sentence answers (core).
        run_command([
            sys.executable,
            'scripts/29_demo_prompt_generation_audit.py',
            '--model-name', MODEL_NAME,
            '--prompt-csv', str(core_prompt_csv),
            '--output-dir', str(exp2_root / 'core'),
            '--max-items', str(MAX_ITEMS),
            '--batch-size', str(BATCH_SIZE_2),
            '--max-new-tokens', str(EXP2_MAX_NEW_TOKENS),
            '--device', DEVICE,
            '--torch-dtype', TORCH_DTYPE,
            '--seed', str(SEED),
        ])

        run_jabber = (core_mode == 'lexically_controlled')
        if run_jabber:
            # 3) Generate full-sentence answers (jabberwocky-prime condition family).
            run_command([
                sys.executable,
                'scripts/29_demo_prompt_generation_audit.py',
                '--model-name', MODEL_NAME,
                '--prompt-csv', str(jabber_prompt_csv),
                '--output-dir', str(exp2_root / 'jabberwocky'),
                '--max-items', str(MAX_ITEMS),
                '--batch-size', str(BATCH_SIZE_2),
                '--max-new-tokens', str(EXP2_MAX_NEW_TOKENS),
                '--device', DEVICE,
                '--torch-dtype', TORCH_DTYPE,
                '--seed', str(SEED),
            ])

        # 4) Automatic structure/thematic-role annotation.
        annotation_inputs = [exp2_root / 'core' / 'item_generations.csv']
        if run_jabber:
            annotation_inputs.append(exp2_root / 'jabberwocky' / 'item_generations.csv')

        annotate_command = [sys.executable, 'scripts/31_annotate_generation_structure_labels.py']
        for input_path in annotation_inputs:
            annotate_command.extend(['--input-csv', str(input_path)])
        run_command(annotate_command)

        # 5) Export strict/lax summaries + unclassified rows for manual review.
        for domain in (['core', 'jabberwocky'] if run_jabber else ['core']):
            domain_root = exp2_root / domain
            item_path = domain_root / 'item_generations.csv'
            if not item_path.exists():
                continue

            frame = pd.read_csv(item_path)

            # Analysis summaries from annotated strict/lax labels.
            strict_summary = summarize_binary(frame, 'generation_class_strict')
            lax_summary = summarize_binary(frame, 'generation_class_lax')
            strict_quality = summarize_quality(frame, 'generation_class_strict')
            lax_quality = summarize_quality(frame, 'generation_class_lax')

            strict_summary.to_csv(domain_root / 'generation_summary_strict.csv', index=False)
            lax_summary.to_csv(domain_root / 'generation_summary_lax.csv', index=False)
            strict_quality.to_csv(domain_root / 'generation_quality_summary_strict.csv', index=False)
            lax_quality.to_csv(domain_root / 'generation_quality_summary_lax.csv', index=False)

            review_mask = (
                frame['generation_class_strict'].astype(str).eq('other')
                | frame['generation_class_lax'].astype(str).eq('other')
            )
            review = frame.loc[review_mask].copy()
            review['needs_review_strict'] = review['generation_class_strict'].astype(str).eq('other')
            review['needs_review_lax'] = review['generation_class_lax'].astype(str).eq('other')

            preferred_columns = [
                'item_index', 'prompt_column', 'prime_condition',
                'target_active', 'target_passive',
                'greedy_answer_first_sentence', 'greedy_answer_first_sentence_normalized',
                'generation_class_detailed', 'generation_structure_reason',
                'argument_structure_inferred', 'role_frame_inferred', 'argument_inference_note',
                'generation_class_strict', 'generation_class_lax',
                'needs_review_strict', 'needs_review_lax',
                'prompt', 'greedy_completion_raw',
            ]
            keep_columns = [col for col in preferred_columns if col in review.columns]
            review = review[keep_columns]

            review_path = exp2_root / f'{domain}_unclassified_for_manual_review.csv'
            review.to_csv(review_path, index=False)
            print(f'Wrote manual-review export: {review_path} (rows={len(review)})')

    for exp2_root in exp2_roots:
        zip_output_dir(exp2_root)
else:
    print('Skipping Experiment 2.')

if RUN_EXPERIMENT_3:
    exp3_base_path = Path(REPO_DIR) / EXPERIMENT3_BASE_CONFIG
    with exp3_base_path.open('r', encoding='utf-8') as handle:
        exp3_config = yaml.safe_load(handle)

    exp3_config.setdefault('experiment', {})
    exp3_config['experiment']['output_dir'] = EXPERIMENT3_OUTPUT_DIR
    exp3_config['experiment']['seed'] = SEED
    exp3_config['experiment']['max_items_per_corpus'] = MAX_ITEMS

    exp3_config['models'] = [
        {
            'name': MODEL_NAME,
            'model_condition': f'{MODEL_TAG}_exp3',
            'device': DEVICE,
            'torch_dtype': TORCH_DTYPE,
            'batch_size': BATCH_SIZE_3,
            'max_length': 1024,
            'local_files_only': False,
            'use_chat_template': False,
            'prompt_style': 'plain',
        }
    ]

    exp3_runtime_config = Path(REPO_DIR) / 'configs' / '_exp3_colab_runtime.yaml'
    with exp3_runtime_config.open('w', encoding='utf-8') as handle:
        yaml.safe_dump(exp3_config, handle, sort_keys=False)

    run_command([
        sys.executable,
        'run_experiment.py',
        '--experiment', 'exp3',
        '--config', str(exp3_runtime_config),
        '--output-dir', EXPERIMENT3_OUTPUT_DIR,
    ])
    zip_output_dir(Path(REPO_DIR) / EXPERIMENT3_OUTPUT_DIR)
else:
    print('Skipping Experiment 3.')

if RUN_EXPERIMENT_4:
    exp4_base_path = Path(REPO_DIR) / EXPERIMENT4_BASE_CONFIG
    with exp4_base_path.open('r', encoding='utf-8') as handle:
        exp4_config = yaml.safe_load(handle)

    exp4_config.setdefault('experiment', {})
    exp4_config['experiment']['output_dir'] = EXPERIMENT4_OUTPUT_DIR
    exp4_config['experiment']['seed'] = SEED
    exp4_config['experiment']['max_items_per_corpus'] = MAX_ITEMS

    exp4_config['models'] = [
        {
            'name': MODEL_NAME,
            'model_condition': f'{MODEL_TAG}_exp4',
            'device': DEVICE,
            'torch_dtype': TORCH_DTYPE,
            'batch_size': BATCH_SIZE_4,
            'max_length': 1024,
            'local_files_only': False,
            'use_chat_template': False,
            'prompt_style': 'plain',
        }
    ]

    exp4_runtime_config = Path(REPO_DIR) / 'configs' / '_exp4_colab_runtime.yaml'
    with exp4_runtime_config.open('w', encoding='utf-8') as handle:
        yaml.safe_dump(exp4_config, handle, sort_keys=False)

    run_command([
        sys.executable,
        'run_experiment.py',
        '--experiment', 'exp4',
        '--config', str(exp4_runtime_config),
        '--output-dir', EXPERIMENT4_OUTPUT_DIR,
    ])
    zip_output_dir(Path(REPO_DIR) / EXPERIMENT4_OUTPUT_DIR)
else:
    print('Skipping Experiment 4.')



In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import pandas as pd

os.chdir(REPO_DIR)

if RUN_EXPERIMENT_1A:
    exp1a_root = Path(REPO_DIR) / EXP1A_OUTPUT_ROOT
    print(f'\nExperiment 1a root: {exp1a_root}')
    if exp1a_root.exists():
        for name in [
            'transitive_item_level_scores.csv',
            'transitive_token_level_scores.csv',
            'transitive_word_level_scores.csv',
            'transitive_item_summary.csv',
            'transitive_token_summary.csv',
            'transitive_word_summary.csv',
            'transitive_report.md',
            'stats/paired_effects.csv',
            'stats/delta_regression_coefficients.csv',
            'stats/lmm_condition_coefficients.csv',
            'stats/analysis_report.md',
        ]:
            path = exp1a_root / name
            if path.exists() and path.suffix == '.csv':
                n_rows = len(pd.read_csv(path))
                print(f'  {name}: exists (rows={n_rows})')
            else:
                print(f'  {name}: exists={path.exists()}')
    else:
        print(f'Skipping missing root: {exp1a_root}')

if RUN_EXPERIMENT_1B:
    for root in (Path(REPO_DIR) / rel_root for rel_root in EXP1B_OUTPUT_ROOTS.values()):
        if root.exists():
            subprocess.run([sys.executable, 'scripts/20_report_choice_runs.py', '--root', str(root)], check=True)
            print(f'Reported: {root}')
        else:
            print(f'Skipping missing root: {root}')

if RUN_EXPERIMENT_2:
    for mode, rel_root in EXP2_AUDIT_OUTPUT_ROOTS.items():
        exp2_root = Path(REPO_DIR) / rel_root
        print(f'\nExperiment 2 audit root [{mode}]: {exp2_root}')
        for domain in (['core', 'jabberwocky'] if mode == 'lexically_controlled' else ['core']):
            item_path = exp2_root / domain / 'item_generations.csv'
            review_path = exp2_root / f'{domain}_unclassified_for_manual_review.csv'
            strict_summary = exp2_root / domain / 'generation_summary_strict.csv'
            lax_summary = exp2_root / domain / 'generation_summary_lax.csv'
            if item_path.exists():
                n_rows = len(pd.read_csv(item_path))
                print(f'  {domain}: item_generations.csv rows={n_rows}')
            else:
                print(f'  {domain}: missing item_generations.csv')
            if review_path.exists():
                n_review = len(pd.read_csv(review_path))
                print(f'  {domain}: manual-review rows={n_review}')
            else:
                print(f'  {domain}: missing manual-review export')
            print(f'  {domain}: strict summary exists={strict_summary.exists()}')
            print(f'  {domain}: lax summary exists={lax_summary.exists()}')

if RUN_EXPERIMENT_3:
    exp3_root = Path(REPO_DIR) / EXPERIMENT3_OUTPUT_DIR
    print(f'\nExperiment 3 output root: {exp3_root}')
    for name in [
        'item_level_results.csv',
        'summary_by_model_condition.csv',
        'summary_by_prime_condition.csv',
        'summary_by_lexicality.csv',
        'summary_by_task.csv',
        'run_metadata.json',
    ]:
        path = exp3_root / name
        if path.exists():
            if path.suffix == '.csv':
                n_rows = len(pd.read_csv(path))
                print(f'  {name}: exists (rows={n_rows})')
            else:
                print(f'  {name}: exists')
        else:
            print(f'  {name}: missing')

if RUN_EXPERIMENT_4:
    exp4_root = Path(REPO_DIR) / EXPERIMENT4_OUTPUT_DIR
    if exp4_root.exists():
        print(f'Experiment 4 output root: {exp4_root}')
    else:
        print(f'Experiment 4 output root not found yet: {exp4_root}')




In [ ]:
from pathlib import Path
import pandas as pd

if RUN_EXPERIMENT_1A:
    exp1a_root = Path(REPO_DIR) / EXP1A_OUTPUT_ROOT
    print(f'\n=== Experiment 1a: {exp1a_root} ===')
    for name in [
        'transitive_item_summary.csv',
        'transitive_token_summary.csv',
        'transitive_word_summary.csv',
        'stats/paired_effects.csv',
        'stats/delta_regression_coefficients.csv',
        'stats/lmm_condition_coefficients.csv',
    ]:
        path = exp1a_root / name
        print(f'\n--- {name}: {path} ---')
        if path.exists():
            display(pd.read_csv(path))
        else:
            print('Not found yet.')

if RUN_EXPERIMENT_1B:
    for mode, root in EXP1B_OUTPUT_ROOTS.items():
        path_root = Path(REPO_DIR) / root
        print(f'\n=== Experiment 1b [{mode}]: {path_root} ===')
        priming_path = path_root / 'priming_framed_results.csv'
        if priming_path.exists():
            display(pd.read_csv(priming_path))
        else:
            print('No priming_framed_results.csv found yet.')

if RUN_EXPERIMENT_2:
    for mode, rel_root in EXP2_AUDIT_OUTPUT_ROOTS.items():
        exp2_root = Path(REPO_DIR) / rel_root
        print(f'\n=== Experiment 2 Generation Audit [{mode}]: {exp2_root} ===')
        for domain in (['core', 'jabberwocky'] if mode == 'lexically_controlled' else ['core']):
            domain_root = exp2_root / domain
            print(f'\n--- {domain}: {domain_root} ---')
            for name in [
                'generation_summary.csv',
                'generation_quality_summary.csv',
                'generation_summary_strict.csv',
                'generation_summary_lax.csv',
                'generation_quality_summary_strict.csv',
                'generation_quality_summary_lax.csv',
            ]:
                path = domain_root / name
                print(f'{name}: {path}')
                if path.exists():
                    display(pd.read_csv(path))
                else:
                    print('Not found yet.')

            review_path = exp2_root / f'{domain}_unclassified_for_manual_review.csv'
            print(f'manual review: {review_path}')
            if review_path.exists():
                display(pd.read_csv(review_path).head(25))
            else:
                print('Not found yet.')

if RUN_EXPERIMENT_3:
    exp3_root = Path(REPO_DIR) / EXPERIMENT3_OUTPUT_DIR
    print(f'\n=== Experiment 3: {exp3_root} ===')
    for name in [
        'summary_by_model_condition.csv',
        'summary_by_prime_condition.csv',
        'summary_by_lexicality.csv',
        'summary_by_task.csv',
    ]:
        path = exp3_root / name
        print(f'\n--- {name}: {path} ---')
        if path.exists():
            display(pd.read_csv(path))
        else:
            print('Not found yet.')

if RUN_EXPERIMENT_4:
    exp4_root = Path(REPO_DIR) / EXPERIMENT4_OUTPUT_DIR
    print(f'\n=== Experiment 4: {exp4_root} ===')
    for name in [
        'summary_by_prime_condition_exp4.csv',
        'summary_by_target_voice_exp4.csv',
        'summary_by_lexicality_exp4.csv',
        'summary_baseline_vs_primed_exp4.csv',
        'ceiling_diagnostics_exp4.csv',
    ]:
        path = exp4_root / name
        print(f'\n--- {name}: {path} ---')
        if path.exists():
            display(pd.read_csv(path))
        else:
            print('Not found yet.')




In [ ]:
from pathlib import Path
import shutil

bundle_root = Path(REPO_DIR) / BASE_OUTPUT_ROOT
archive_stem = Path('/content') / f'colab_{MODEL_TAG}_results'
archive_path = shutil.make_archive(
    str(archive_stem),
    'zip',
    root_dir=str(bundle_root.parent),
    base_dir=str(bundle_root.name),
)
print('Created:', archive_path)
